This notebook Analyze whether dental and vision coverage increase plan costs.

In [2]:
# Plans With Dental or Vision Coverage Cost More
# 1- Do plans with dental cost more than plans without?
# 2- Do plans with vision cost more than plans without?
# 3- Does this change by metal level (Bronze, Silver, Gold)?

In [3]:
import pandas as pd   
import numpy as np  
import matplotlib.pyplot as plt  
import seaborn as sns
from datetime import datetime

In [4]:
df_rates = pd.read_csv('../data/Rate_PUF.csv',low_memory=False)

In [5]:
df_rates.head()

,BusinessYear,StateCode,IssuerId,SourceName,ImportDate,RateEffectiveDate,RateExpirationDate,PlanId,RatingAreaId,Tobacco,Age,IndividualRate,IndividualTobaccoRate,Couple,PrimarySubscriberAndOneDependent,PrimarySubscriberAndTwoDependents,PrimarySubscriberAndThreeOrMoreDependents,CoupleAndOneDependent,CoupleAndTwoDependents,CoupleAndThreeOrMoreDependents
0,2026,AK,21989,HIOS,2025-10-15,2026-01-01,2026-12-31,21989AK0030001,Rating Area 1,No Preference,0-14,65.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026,AK,21989,HIOS,2025-10-15,2026-01-01,2026-12-31,21989AK0030001,Rating Area 1,No Preference,15,65.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026,AK,21989,HIOS,2025-10-15,2026-01-01,2026-12-31,21989AK0030001,Rating Area 1,No Preference,16,65.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026,AK,21989,HIOS,2025-10-15,2026-01-01,2026-12-31,21989AK0030001,Rating Area 1,No Preference,17,65.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026,AK,21989,HIOS,2025-10-15,2026-01-01,2026-12-31,21989AK0030001,Rating Area 1,No Preference,18,65.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df_rates_plans = df_rates[['BusinessYear', 'StateCode', 'IssuerId', 'PlanId', 'Age', 'IndividualRate']]
df_rates_plans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2235761 entries, 0 to 2235760
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   BusinessYear    int64  
 1   StateCode       object 
 2   IssuerId        int64  
 3   PlanId          object 
 4   Age             object 
 5   IndividualRate  float64
dtypes: float64(1), int64(2), object(3)
memory usage: 102.3+ MB


In [7]:
df_rates_plans.columns = (df_rates_plans.columns.str.lower().str.strip().str.replace(" ", "_"))
df_rates_plans.head()

,businessyear,statecode,issuerid,planid,age,individualrate
0,2026,AK,21989,21989AK0030001,0-14,65.0
1,2026,AK,21989,21989AK0030001,15,65.0
2,2026,AK,21989,21989AK0030001,16,65.0
3,2026,AK,21989,21989AK0030001,17,65.0
4,2026,AK,21989,21989AK0030001,18,65.0


In [8]:
df_rates_plans = df_rates_plans[df_rates_plans['businessyear'] == 2026] 
df_rates_plans

,businessyear,statecode,issuerid,planid,age,individualrate
0,2026,AK,21989,21989AK0030001,0-14,65.00
1,2026,AK,21989,21989AK0030001,15,65.00
2,2026,AK,21989,21989AK0030001,16,65.00
3,2026,AK,21989,21989AK0030001,17,65.00
4,2026,AK,21989,21989AK0030001,18,65.00
...,...,...,...,...,...,...
2235756,2026,WY,83964,83964WY0040002,60,44.57
2235757,2026,WY,83964,83964WY0040002,61,44.57
2235758,2026,WY,83964,83964WY0040002,62,44.57
2235759,2026,WY,83964,83964WY0040002,63,44.57


In [9]:
df_rates_plans = df_rates_plans[df_rates_plans['age'] == '27']
df_rates_plans

,businessyear,statecode,issuerid,planid,age,individualrate
13,2026,AK,21989,21989AK0030001,27,35.00
64,2026,AK,21989,21989AK0030001,27,35.00
115,2026,AK,21989,21989AK0030001,27,35.00
166,2026,AK,21989,21989AK0050001,27,35.00
217,2026,AK,21989,21989AK0050001,27,35.00
...,...,...,...,...,...,...
2235519,2026,WY,83964,83964WY0040001,27,29.98
2235570,2026,WY,83964,83964WY0040001,27,29.98
2235621,2026,WY,83964,83964WY0040002,27,37.34
2235672,2026,WY,83964,83964WY0040002,27,37.34


In [10]:
df_attributes = pd.read_csv('../data/plan_attributes_PUF.csv',low_memory=False)

In [11]:
df_attributes.head()

,BusinessYear,StateCode,IssuerId,IssuerMarketPlaceMarketingName,SourceName,ImportDate,MarketCoverage,DentalOnlyPlan,StandardComponentId,PlanMarketingName,...,TEHBDedOutOfNetFamilyPerPerson,TEHBDedOutOfNetFamilyPerGroup,TEHBDedCombInnOonIndividual,TEHBDedCombInnOonFamilyPerPerson,TEHBDedCombInnOonFamilyPerGroup,IsHSAEligible,HSAOrHRAEmployerContribution,HSAOrHRAEmployerContributionAmount,URLForSummaryofBenefitsCoverage,PlanBrochure
0,2026,AK,21989,Delta Dental of Alaska,HIOS,10/15/2025,Individual,Yes,21989AK0030001,Delta Dental Premier Plan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.deltadentalak.com/-/media/deltaden...
1,2026,AK,21989,Delta Dental of Alaska,HIOS,10/15/2025,Individual,Yes,21989AK0030001,Delta Dental Premier Plan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.deltadentalak.com/-/media/deltaden...
2,2026,AK,21989,Delta Dental of Alaska,HIOS,10/15/2025,Individual,Yes,21989AK0050001,Delta Dental PPO 1000 Plan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.deltadentalak.com/-/media/deltaden...
3,2026,AK,21989,Delta Dental of Alaska,HIOS,10/15/2025,Individual,Yes,21989AK0050001,Delta Dental PPO 1000 Plan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.deltadentalak.com/-/media/deltaden...
4,2026,AK,21989,Delta Dental of Alaska,HIOS,10/15/2025,Individual,Yes,21989AK0050002,Delta Dental PPO 1500 Plan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.deltadentalak.com/-/media/deltaden...


In [12]:
df_attr_plans = df_attributes[['BusinessYear', 'StateCode', 'IssuerId', 'IssuerMarketPlaceMarketingName', 'MarketCoverage', 'DentalOnlyPlan',
                     'StandardComponentId', 'PlanMarketingName', 'MetalLevel', 'IsHSAEligible']]
df_attr_plans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22059 entries, 0 to 22058
Data columns (total 10 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   BusinessYear                    22059 non-null  int64 
 1   StateCode                       22059 non-null  object
 2   IssuerId                        22059 non-null  int64 
 3   IssuerMarketPlaceMarketingName  22059 non-null  object
 4   MarketCoverage                  22059 non-null  object
 5   DentalOnlyPlan                  22059 non-null  object
 6   StandardComponentId             22059 non-null  object
 7   PlanMarketingName               22059 non-null  object
 8   MetalLevel                      22059 non-null  object
 9   IsHSAEligible                   20670 non-null  object
dtypes: int64(2), object(8)
memory usage: 1.7+ MB


In [13]:
df_attr_plans.columns = (df_attr_plans.columns.str.lower().str.strip().str.replace(" ", "_"))
df_attr_plans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22059 entries, 0 to 22058
Data columns (total 10 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   businessyear                    22059 non-null  int64 
 1   statecode                       22059 non-null  object
 2   issuerid                        22059 non-null  int64 
 3   issuermarketplacemarketingname  22059 non-null  object
 4   marketcoverage                  22059 non-null  object
 5   dentalonlyplan                  22059 non-null  object
 6   standardcomponentid             22059 non-null  object
 7   planmarketingname               22059 non-null  object
 8   metallevel                      22059 non-null  object
 9   ishsaeligible                   20670 non-null  object
dtypes: int64(2), object(8)
memory usage: 1.7+ MB


In [14]:
df_attr_plans = df_attr_plans[(df_attr_plans['businessyear'] == 2026) & 
                (df_attr_plans['marketcoverage'] == 'Individual')]
df_attr_plans.head()

,businessyear,statecode,issuerid,issuermarketplacemarketingname,marketcoverage,dentalonlyplan,standardcomponentid,planmarketingname,metallevel,ishsaeligible
0,2026,AK,21989,Delta Dental of Alaska,Individual,Yes,21989AK0030001,Delta Dental Premier Plan,Low,NaN
1,2026,AK,21989,Delta Dental of Alaska,Individual,Yes,21989AK0030001,Delta Dental Premier Plan,Low,NaN
2,2026,AK,21989,Delta Dental of Alaska,Individual,Yes,21989AK0050001,Delta Dental PPO 1000 Plan,High,NaN
3,2026,AK,21989,Delta Dental of Alaska,Individual,Yes,21989AK0050001,Delta Dental PPO 1000 Plan,High,NaN
4,2026,AK,21989,Delta Dental of Alaska,Individual,Yes,21989AK0050002,Delta Dental PPO 1500 Plan,High,NaN
